In [4]:
import rasterio

with rasterio.open('dem_maps/norcoast_dem.tif') as src:
    print(src.crs)
    print(src.bounds)

EPSG:26910
BoundingBox(left=549994.000025334, bottom=4229994.000030626, right=560006.000025334, top=4240006.000030626)


In [ ]:
from rasterio.warp import transform_bounds

with rasterio.open('dem_maps/norcoast_b23_dem.tif') as src:
    # Your current bounds (UTM)
    bounds_utm = src.bounds
    print(f"UTM Bounds: {bounds_utm}")
    
    # Transform to WGS84 for OSM
    bounds_wgs84 = transform_bounds('EPSG:26910', 'EPSG:4326', *bounds_utm)
    west, south, east, north = bounds_wgs84
    
    print(f"\nBounding Box for OSM (WGS84 lat/lon):")
    print(f"West:  {west:.8f}")
    print(f"South: {south:.8f}")
    print(f"East:  {east:.8f}")
    print(f"North: {north:.8f}")
    
    # Format for Overpass API
    bbox_overpass = f"{south},{west},{north},{east}"
    print(f"\nOverpass API format: {bbox_overpass}")

UTM Bounds: BoundingBox(left=549994.000025334, bottom=4229994.000030626, right=560006.000025334, top=4240006.000030626)

Bounding Box for OSM (WGS84 lat/lon):
West:  -122.42889385
South: 38.21591735
East:  -122.31368042
North: 38.30675808

Overpass API format: 38.215917346983794,-122.42889385471949,38.30675808340658,-122.31368041640931


In [10]:
import osmnx as ox
print(ox.__version__)

2.1.0


In [17]:
import osmnx as ox
import os

# Create osm_data folder if it doesn't exist
os.makedirs("osm_data", exist_ok=True)

# Your coordinates
# bbox format: (left, bottom, right, top) = (west, south, east, north)
west = -122.42889385
south = 38.21591735
east = -122.31368042
north = 38.30675808

bbox = (west, south, east, north)

print(f"Bbox: (west={west}, south={south}, east={east}, north={north})")

# Download buildings
print("\nDownloading buildings...")
buildings = ox.features_from_bbox(bbox, tags={'building': True})
buildings.to_file("osm_data/norcoast_b23_buildings.geojson", driver='GeoJSON')
print(f"Downloaded {len(buildings)} buildings")

# Download roads
print("\nDownloading roads...")
roads = ox.graph_from_bbox(bbox, network_type='all')
# Save as GeoPackage (modern format, single file)
ox.save_graph_geopackage(roads, filepath='osm_data/norcoast_b23_roads.gpkg')
print("Roads downloaded and saved as GeoPackage")

# Download other features (landuse, natural features, etc.)
print("\nDownloading other features...")
other_features = ox.features_from_bbox(bbox,
                                       tags={'landuse': True,
                                             'natural': True,
                                             'waterway': True,
                                             'amenity': True})
other_features.to_file("osm_data/norcoast_b23_features.geojson", driver='GeoJSON')
print(f"Downloaded {len(other_features)} other features")

print("\nAll OSM data saved to 'osm_data/' folder")

Bbox: (west=-122.42889385, south=38.21591735, east=-122.31368042, north=38.30675808)

Downloaded 3111 buildings

Roads downloaded and saved as GeoPackage

Downloaded 1138 other features

All OSM data saved to 'osm_data/' folder
